# Notebook 6: Regulatory Capital & Portfolio Risk Report
## FRTB Capital Summary | Risk Attribution | Final Dashboard

**FRTB (Fundamental Review of the Trading Book):**
- Basel IV replaces VaR with **Expected Shortfall (ES)** at 97.5%
- Introduces **Liquidity Horizons**: 10/20/40/60/120 days per asset class
- Requires P&L Attribution Test and Backtest to qualify for IMA


In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from datetime import datetime

BASE_DIR    = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
DATA_DIR    = os.path.join(BASE_DIR, 'data')
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')

log_returns       = pd.read_csv(os.path.join(DATA_DIR, 'log_returns.csv'), index_col=0, parse_dates=True)
portfolio_returns = pd.read_csv(os.path.join(DATA_DIR, 'portfolio_returns.csv'), index_col=0, parse_dates=True)['portfolio_return']
weights           = pd.read_csv(os.path.join(DATA_DIR, 'portfolio_weights.csv'), index_col=0)['weight']
var_estimates     = pd.read_csv(os.path.join(DATA_DIR, 'var_estimates.csv'))
backtest          = pd.read_csv(os.path.join(DATA_DIR, 'backtest_results.csv'), parse_dates=['date'])

CONFIDENCE_LEVEL = 0.99
ALPHA            = 1 - CONFIDENCE_LEVEL
PORTFOLIO_VALUE  = 10_000_000
print('All data loaded.')

In [ ]:
# ── 1. FRTB Expected Shortfall by Liquidity Horizon ──────────────────────────
FRTB_HORIZONS = {
    'SPY':       10,
    'EURUSD':    10,
    'GLD':       20,
    'EEM':       20,
    'Crude_Oil': 20,
}

FRTB_CONF = 0.975

es_by_asset = {}
for asset, horizon in FRTB_HORIZONS.items():
    if asset not in log_returns.columns:
        continue
    r      = log_returns[asset].iloc[-250:]
    alpha  = 1 - FRTB_CONF
    var_1d = np.percentile(r, alpha * 100)
    es_1d  = r[r <= var_1d].mean()
    es_Nd  = es_1d * np.sqrt(horizon)
    wt     = weights.get(asset, 0.0)
    es_by_asset[asset] = {
        'Liquidity Horizon': horizon,
        'ES 1D (97.5%)': es_1d,
        f'ES ND (97.5%)': es_Nd,
        'Capital ($)': abs(es_Nd) * wt * PORTFOLIO_VALUE
    }

frtb_df = pd.DataFrame(es_by_asset).T
print('=== FRTB Expected Shortfall by Asset ===')
print(frtb_df.round(4).to_string())
print(f'\nTotal FRTB Capital: ${frtb_df["Capital ($)"].sum():,.0f}')

In [ ]:
# ── 2. Full Basel III Capital Charge Summary ──────────────────────────────────
MULTIPLIER   = 3.0
LOOKBACK     = 250
avg_var      = np.percentile(portfolio_returns.iloc[-LOOKBACK:].values, ALPHA*100)
var_component = max(abs(avg_var), MULTIPLIER * abs(avg_var)) * np.sqrt(10)

gfc_mask      = (portfolio_returns.index >= '2008-09-01') & (portfolio_returns.index <= '2009-09-01')
gfc_returns   = portfolio_returns[gfc_mask]
gfc_var       = np.percentile(gfc_returns.values, ALPHA*100)
svar_component = max(abs(gfc_var), MULTIPLIER * abs(gfc_var)) * np.sqrt(10)

total_capital = (var_component + svar_component) * PORTFOLIO_VALUE
frtb_capital  = frtb_df['Capital ($)'].sum()

capital_summary = pd.DataFrame([
    ['Basel III VaR Component (99%, 10D)', f'${var_component*PORTFOLIO_VALUE:,.0f}',
     f'{var_component:.4f}'],
    ['Basel III SVaR Component (GFC)',     f'${svar_component*PORTFOLIO_VALUE:,.0f}',
     f'{svar_component:.4f}'],
    ['Basel III Total MRC',               f'${total_capital:,.0f}',
     f'{total_capital/PORTFOLIO_VALUE:.2%} of portfolio'],
    ['FRTB IMA ES Capital (97.5%)',        f'${frtb_capital:,.0f}',
     f'{frtb_capital/PORTFOLIO_VALUE:.2%} of portfolio'],
], columns=['Component', 'Capital ($)', 'Note'])

print('=== REGULATORY CAPITAL SUMMARY ===')
print(capital_summary.to_string(index=False))

In [ ]:
# ── 3. Risk Attribution Dashboard ────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Panel 1: Capital allocation pie
ax1 = fig.add_subplot(gs[0, 0])
cap_vals = frtb_df['Capital ($)'].values
ax1.pie(cap_vals, labels=frtb_df.index, autopct='%1.1f%%', startangle=90,
        colors=['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd'])
ax1.set_title('FRTB Capital\nAllocation by Asset', fontweight='bold', fontsize=9)

# Panel 2: VaR method comparison
ax2  = fig.add_subplot(gs[0, 1])
methods = var_estimates['method'].str.replace('_', ' ').str.title()
var10d  = abs(var_estimates['var_10d'].values)
es10d   = abs(var_estimates['es_10d'].values)
x = np.arange(len(methods))
ax2.bar(x - 0.2, var10d * PORTFOLIO_VALUE / 1e6, width=0.4, label='VaR 10D', color='steelblue')
ax2.bar(x + 0.2, es10d  * PORTFOLIO_VALUE / 1e6, width=0.4, label='ES 10D',  color='salmon')
ax2.set_xticks(x)
ax2.set_xticklabels(methods, rotation=15, ha='right', fontsize=8)
ax2.set_ylabel('$M')
ax2.set_title('VaR vs ES by Method\n($10M portfolio)', fontweight='bold', fontsize=9)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, axis='y')

# Panel 3: Component VaR
ax3 = fig.add_subplot(gs[0, 2])
cov_mat = log_returns.iloc[-250:].cov()
w       = weights.reindex(log_returns.columns).fillna(0).values
port_var_scalar = float(w @ cov_mat.values @ w)
marginal_var    = (cov_mat.values @ w) / np.sqrt(port_var_scalar) * norm.ppf(ALPHA)
component_var   = marginal_var * w
cv_df     = pd.Series(component_var, index=log_returns.columns)
cv_df_pct = cv_df / (cv_df.sum() + 1e-12) * 100
colors_cv = ['#d62728' if v > 0 else '#2ca02c' for v in cv_df_pct]
cv_df_pct.plot(kind='bar', ax=ax3, color=colors_cv, alpha=0.85)
ax3.set_title('Component VaR\nContribution (%)', fontweight='bold', fontsize=9)
ax3.set_ylabel('%')
ax3.tick_params(axis='x', rotation=30)
ax3.grid(alpha=0.3, axis='y')

# Panel 4: P&L histogram
ax4 = fig.add_subplot(gs[1, :])
ax4.hist(portfolio_returns.values * PORTFOLIO_VALUE / 1e6, bins=100,
         density=False, color='steelblue', alpha=0.6, label='Daily P&L ($M)')
hs_var_val = np.percentile(portfolio_returns.iloc[-250:], ALPHA*100)
hs_es_val  = portfolio_returns.iloc[-250:][portfolio_returns.iloc[-250:] <= hs_var_val].mean()
ax4.axvline(hs_var_val * PORTFOLIO_VALUE / 1e6, color='red',     lw=2,
            label=f'VaR 99% (HS): ${hs_var_val*PORTFOLIO_VALUE/1e6:.2f}M')
ax4.axvline(hs_es_val  * PORTFOLIO_VALUE / 1e6, color='darkred', lw=2, linestyle='--',
            label=f'ES 99% (HS): ${hs_es_val*PORTFOLIO_VALUE/1e6:.2f}M')
ax4.set_xlabel('Daily P&L ($M)')
ax4.set_ylabel('Frequency')
ax4.set_title('Portfolio Daily P&L Distribution with VaR/ES Risk Measures', fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

# Panel 5: Cumulative P&L + drawdown
ax5  = fig.add_subplot(gs[2, :])
ax5b = ax5.twinx()
cum_pnl  = (1 + portfolio_returns).cumprod()
drawdown = cum_pnl / cum_pnl.cummax() - 1
ax5.plot(cum_pnl.index,   cum_pnl.values,   color='steelblue', lw=1.5, label='Portfolio Growth')
ax5b.fill_between(drawdown.index, drawdown.values, 0, color='salmon', alpha=0.4, label='Drawdown')
ax5b.set_ylabel('Drawdown', color='red')
ax5.set_ylabel('Portfolio Value (Base=1)', color='steelblue')
ax5.set_title('Cumulative Portfolio Performance & Drawdown', fontweight='bold')
ax5.legend(loc='upper left')
ax5b.legend(loc='lower right')
ax5.grid(alpha=0.3)

fig.suptitle(
    f'Market Risk Engine - Portfolio Risk Dashboard\n'
    f'Portfolio Value: $10M | Confidence: 99% | Report Date: {datetime.today().strftime("%Y-%m-%d")}',
    fontsize=12, fontweight='bold', y=0.98
)

plt.savefig(os.path.join(REPORTS_DIR, 'fig10_risk_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Risk dashboard saved.')

In [ ]:
# ── 4. Final Summary Report ───────────────────────────────────────────────────
print('=' * 65)
print('        MARKET RISK ENGINE - PROJECT SUMMARY')
print('=' * 65)
print(f'\nPortfolio : 5-asset multi-class (Equity, Gold, EM, FX, Oil)')
print(f'Period    : {portfolio_returns.index[0].date()} to {portfolio_returns.index[-1].date()}')
print(f'Obs       : {len(portfolio_returns):,} trading days')

ann_vol = portfolio_returns.std() * np.sqrt(252)
ann_ret = portfolio_returns.mean() * 252
sharpe  = ann_ret / ann_vol
max_dd  = (((1+portfolio_returns).cumprod()) / ((1+portfolio_returns).cumprod().cummax()) - 1).min()

print(f'\n--- Portfolio Statistics ---')
print(f'Annualized Return     : {ann_ret:.2%}')
print(f'Annualized Volatility : {ann_vol:.2%}')
print(f'Sharpe Ratio          : {sharpe:.4f}')
print(f'Maximum Drawdown      : {max_dd:.2%}')

print(f'\n--- VaR Estimates (99%, 10-Day, $10M) ---')
for _, row in var_estimates.iterrows():
    print(f"{row['method']:15s}: VaR = ${abs(row['var_10d'])*PORTFOLIO_VALUE:>10,.0f} | ES = ${abs(row['es_10d'])*PORTFOLIO_VALUE:>10,.0f}")

print(f'\n--- Regulatory Capital ---')
print(f'Basel III MRC         : ${total_capital:,.0f} ({total_capital/PORTFOLIO_VALUE:.2%})')
print(f'FRTB IMA ES Capital   : ${frtb_capital:,.0f} ({frtb_capital/PORTFOLIO_VALUE:.2%})')

exc_count = backtest['hs_exception'].sum()
zone = 'GREEN' if exc_count <= 4 else ('YELLOW' if exc_count <= 9 else 'RED')
print(f'\n--- Model Validation ---')
print(f'Backtest Zone         : {zone} ({exc_count} exceptions / 250 days)')
print(f'\n✅ All notebooks completed. Project ready for GitHub.')